Classificacao de Instrumentos

In [23]:
import librosa
import numpy as np
import os
import re
from tqdm import tqdm
from pathlib import Path

from scipy.signal import convolve2d
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

1. Carregamento

In [24]:
def carregar_audio(caminho_arquivo, sr=22050):
    y, _ = librosa.load(caminho_arquivo, sr=sr)

    # normalizacao de amplitude
    y = y / (np.max(np.abs(y)) + 1e-8)
    return y

# Mapeia instrumentos para labels
CLASSES = {'vio': 0, 'pia': 1, 'cla': 2}
BASE_PATH = Path("datasets")

In [25]:
BASE_PATH

PosixPath('datasets')

In [26]:
def nome_seguro(stem: str) -> str:
    # Extrai tags entre []: [vio][cla] -> vio, cla
    tags = re.findall(r"\[([^\]]+)\]", stem)

    # Remove as tags e limpa o restante
    resto = re.sub(r"\[[^\]]+\]", "", stem).strip("_")

    partes = [p for p in (*tags, resto) if p]
    novo = "_".join(partes)

    # Mantém só caracteres seguros
    novo = re.sub(r"[^A-Za-z0-9_-]+", "_", novo)
    novo = re.sub(r"_+", "_", novo).strip("_")

    return novo or "audio"

renomeados = []

for old_path in BASE_PATH.rglob("*.wav"):
    new_stem = nome_seguro(old_path.stem)
    new_path = old_path.with_name(f"{new_stem}{old_path.suffix.lower()}")

    # Evita colisão de nomes
    i = 1
    while new_path.exists() and new_path != old_path:
        new_path = old_path.with_name(f"{new_stem}_{i}{old_path.suffix.lower()}")
        i += 1

    if new_path != old_path:
        old_path.replace(new_path)
        renomeados.append((old_path, new_path))

print(f"Arquivos renomeados: {len(renomeados)}")
for old, new in renomeados[:10]:
    print(f"{old.name} -> {new.name}")

Arquivos renomeados: 0


In [27]:
;;;;;

<>:1: SyntaxWarning: 'str' object is not callable; perhaps you missed a comma?
<>:1: SyntaxWarning: 'str' object is not callable; perhaps you missed a comma?
/tmp/ipykernel_43472/1341573423.py:1: SyntaxWarning: 'str' object is not callable; perhaps you missed a comma?
  ("")("")("")("")("")


TypeError: 'str' object is not callable

In [ ]:
# lista apenas com os caminhos renomeados
arquivos_renomeados = [new for _, new in renomeados]

print(f"Total renomeados: {len(arquivos_renomeados)}")
print("Exemplos:")
for p in arquivos_renomeados[:5]:
    print(p)

Total renomeados: 6705
Exemplos:
datasets/irmas-training-data/IRMAS-TrainingData/gel/gel_dru_jaz_blu_168_0826_1.wav
datasets/irmas-training-data/IRMAS-TrainingData/gel/gel_dru_pop_roc_034_0965_3.wav
datasets/irmas-training-data/IRMAS-TrainingData/gel/gel_dru_pop_roc_224_0843_1.wav
datasets/irmas-training-data/IRMAS-TrainingData/gel/gel_dru_pop_roc_145_0949_1.wav
datasets/irmas-training-data/IRMAS-TrainingData/gel/gel_pop_roc_0914_3.wav


In [ ]:
path

'datasets\x0bio\\[vio][cla]2083__1.wav'

In [ ]:
y, sr = librosa.load(BASE_PATH / "vio" / "[vio][cla]2095__2.wav", sr=22050)

/tmp/ipykernel_43472/1550169911.py:1: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(BASE_PATH / "vio" / "[vio][cla]2095__2.wav", sr=22050)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


FileNotFoundError: [Errno 2] No such file or directory: 'datasets/vio/[vio][cla]2095__2.wav'

In [ ]:


# STFT → espectrograma de magnitude
D = librosa.stft(y, n_fft=2048, hop_length=512)
S_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)

# Isso é agora uma IMAGEM 2D: eixo X = tempo, eixo Y = frequência
print(S_db.shape) 

/tmp/ipykernel_30364/1051480559.py:1: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(BASE_PATH / "vio" / "[vio][cla]2095__2.wav", sr=22050)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


FileNotFoundError: [Errno 2] No such file or directory: 'datasets/vio/[vio][cla]2095__2.wav'

In [ ]:
path = BASE_PATH / "vio" / "[vio][cla]2095__2.wav"



SyntaxError: invalid decimal literal (67737905.py, line 2)

In [ ]:
print(path.exists())

False


In [ ]:
from pydub import AudioSegment

audio = AudioSegment.from_file(path)
#audio.export("datasets/vio/[vio][cla]2095__2_fixed.wav", format="wav", parameters=["-acodec", "pcm_s16le"])


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\kamil\\OneDrive\\Área de Trabalho\\academico\\aulas_facul\\P8\\pdi\\projeto final\\instrument-classification\\datasets\\vio\\[vio][cla]2095__2.wav'

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\kamil\\OneDrive\\Área de Trabalho\\academico\\aulas_facul\\P8\\pdi\\projeto final\\instrument-classification\\datasets\\vio\\[vio][cla]2095__2.wav'

In [ ]:
y, sr = librosa.load("datasets/vio/[vio][cla]2095__2_fixed.wav", sr=22050)


/tmp/ipykernel_30364/1141139659.py:1: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load("datasets/vio/[vio][cla]2095__2_fixed.wav", sr=22050)


FileNotFoundError: [Errno 2] No such file or directory: 'datasets/vio/[vio][cla]2095__2_fixed.wav'

2. Extracao de Features